In [0]:
landing_path = "/Volumes/factory_catalog/bronze/raw_landing/"

display(dbutils.fs.ls(landing_path))


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timedelta
import random

random.seed(42)

machines = [f"M{i:03d}" for i in range(1, 21)]

records = []
start_time = datetime.now() - timedelta(hours=24)

for i in range(500):
    machine_id = random.choice(machines)
    event_time = start_time + timedelta(minutes=random.randint(0, 1440))

    records.append((
        machine_id,
        event_time,
        round(random.uniform(25, 60), 2),
        round(random.uniform(60, 100), 2)
    ))

sensor_df = spark.createDataFrame(
    records,
    ["machine_id", "event_time", "temperature", "utilization_pct"]
)

display(sensor_df.limit(10))


In [0]:

from pyspark.sql import functions as F

# Add duplicate records
duplicate_records = sensor_df.limit(10)

# Add records with missing values
bad_records = [
    ("M001", None, 45.5, 85.0),
    ("M002", datetime.now(), None, 75.0),
    ("M003", datetime.now(), 42.0, None)
]

bad_df = spark.createDataFrame(
    bad_records,
    ["machine_id", "event_time", "temperature", "utilization_pct"]
)

# Combine normal data, duplicates, and bad records
sensor_df_with_bad_data = (
    sensor_df
    .unionByName(duplicate_records)
    .unionByName(bad_df)
)

print("Original records:", sensor_df.count())
print("Duplicate records added:", duplicate_records.count())
print("Bad records added:", bad_df.count())
print("Total records:", sensor_df_with_bad_data.count())

display(sensor_df_with_bad_data.orderBy(F.col("event_time").asc_nulls_first()).limit(20))


In [0]:

from datetime import timedelta

now = datetime.now()

late_records = [
    ("M004", now - timedelta(hours=48), 47.5, 82.0),
    ("M005", now - timedelta(hours=36), 51.2, 76.0),
    ("M006", now - timedelta(hours=30), 39.8, 91.0)
]

late_df = spark.createDataFrame(
    late_records,
    ["machine_id", "event_time", "temperature", "utilization_pct"]
)

sensor_df_with_bad_data = (
    sensor_df_with_bad_data
    .unionByName(late_df)
)

print("Late records added:", late_df.count())
print("Total records:", sensor_df_with_bad_data.count())

display(late_df)


In [0]:

bronze_path = "/Volumes/factory_catalog/bronze/raw_landing/sensor_readings"

(
    sensor_df_with_bad_data
    .withColumn("ingestion_time", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

print("Sensor data saved to Bronze landing storage!")


In [0]:

saved_sensor_df = spark.read.format("delta").load(bronze_path)

print("Records saved:", saved_sensor_df.count())

display(saved_sensor_df.limit(10))


In [0]:

from pyspark.sql import functions as F

# Read the raw sensor data from Bronze landing storage
raw_sensor_df = spark.read.format("delta").load(bronze_path)

# Remove records with missing or invalid values
clean_sensor_df = raw_sensor_df.filter(
    F.col("machine_id").isNotNull()
    & F.col("event_time").isNotNull()
    & F.col("temperature").isNotNull()
    & F.col("utilization_pct").isNotNull()
    & F.col("temperature").between(0, 100)
    & F.col("utilization_pct").between(0, 100)
)

# Remove duplicate sensor records
clean_sensor_df = clean_sensor_df.dropDuplicates([
    "machine_id",
    "event_time",
    "temperature",
    "utilization_pct"
])

print("Raw records:", raw_sensor_df.count())
print("Clean records:", clean_sensor_df.count())
print("Removed records:", raw_sensor_df.count() - clean_sensor_df.count())

display(clean_sensor_df.limit(10))


In [0]:

silver_table = "factory_catalog.silver.sensor_readings"

(
    clean_sensor_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Clean sensor data saved to Silver!")


In [0]:

silver_df = spark.table("factory_catalog.silver.sensor_readings")

print("Silver table records:", silver_df.count())

display(silver_df.limit(10))


In [0]:

from pyspark.sql import functions as F

gold_df = (
    silver_df
    .groupBy("machine_id")
    .agg(
        F.count("*").alias("total_readings"),
        F.round(F.avg("temperature"), 2).alias("avg_temperature"),
        F.round(F.max("temperature"), 2).alias("max_temperature"),
        F.round(F.avg("utilization_pct"), 2).alias("avg_utilization_pct"),
        F.round(F.min("event_time").cast("long"), 0).alias("first_reading_epoch"),
        F.round(F.max("event_time").cast("long"), 0).alias("last_reading_epoch")
    )
)

gold_table = "factory_catalog.gold.sensor_summary"

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print("Gold sensor summary created!")
display(spark.table(gold_table).orderBy("machine_id"))


In [0]:

# Check the record counts in each layer

bronze_count = spark.read.format("delta").load(bronze_path).count()

silver_count = spark.table(
    "factory_catalog.silver.sensor_readings"
).count()

gold_count = spark.table(
    "factory_catalog.gold.sensor_summary"
).count()

print("BRONZE records:", bronze_count)
print("SILVER records:", silver_count)
print("GOLD machine summaries:", gold_count)


In [0]:

display(spark.sql("SHOW TABLES IN factory_catalog.bronze"))
display(spark.sql("SHOW TABLES IN factory_catalog.silver"))
display(spark.sql("SHOW TABLES IN factory_catalog.gold"))


In [0]:

from datetime import datetime, timedelta
import random

random.seed(100)

products = ["P001", "P002", "P003", "P004", "P005"]
orders = []

for i in range(1, 101):
    order_date = datetime.now() - timedelta(
        days=random.randint(0, 30)
    )

    orders.append((
        f"ORD{i:04d}",
        random.choice(products),
        random.randint(50, 1000),
        order_date,
        random.choice(["Pending", "In Progress", "Completed"])
    ))

orders_df = spark.createDataFrame(
    orders,
    [
        "order_id",
        "product_id",
        "quantity",
        "order_date",
        "status"
    ]
)

print("Production orders generated:", orders_df.count())
display(orders_df.limit(10))


In [0]:

orders_bronze_table = "factory_catalog.bronze.production_orders"

(
    orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(orders_bronze_table)
)

print("Production orders saved to Bronze!")

display(spark.table(orders_bronze_table).limit(10))


In [0]:

print(
    "Production orders in Bronze:",
    spark.table("factory_catalog.bronze.production_orders").count()
)
